In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [14]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [15]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=tools,
    system_prompt=prompt
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [20]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='1e636f6b-bd71-4c66-800f-9450b897190e'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '680bkatjf', 'function': {'arguments': '{"query":"langchain-mcp-adapters library"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 355, 'total_tokens': 375, 'completion_time': 0.031957218, 'completion_tokens_details': None, 'prompt_time': 0.02397106, 'prompt_tokens_details': None, 'queue_time': 0.005830015, 'total_time': 0.055928278}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd72f-0699-7390-93f6-edd0ae1ef37a-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '680b

## Online MCP

In [21]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=tools,
)

In [22]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='8c9edf35-5528-4fa0-9760-65b3af7540ab'),
              AIMessage(content="I'm sorry, I can only answer questions about LangChain, LangGraph and LangSmith.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 349, 'total_tokens': 369, 'completion_time': 0.03174379, 'completion_tokens_details': None, 'prompt_time': 0.023160642, 'prompt_tokens_details': None, 'queue_time': 0.016711857, 'total_time': 0.054904432}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd72f-a0a8-7092-9f17-09742d96cb63-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 349, 'output_tokens': 20, 'total_tokens': 369})]}
